# Setup

In [1]:
!pip install -q langchain langchain-openai langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.1 MB/s eta 0:00:00


- Instalowane pakiety to:
  - `langchain` - główna biblioteka LangChain, która zapewnia narzędzia do budowania aplikacji opartych na modelach językowych
  - `langchain-openai` - integracja LangChain z modelami OpenAI (takimi jak GPT)
  - `langchain_experimental` - moduł zawierający eksperymentalne funkcjonalności LangChain, które nie są jeszcze częścią głównej biblioteki

In [ ]:
import os
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.agents import AgentType
from langchain_openai import ChatOpenAI
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from google.colab import userdata

```python
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
```
To funkcja z eksperymentalnego modułu LangChain, która tworzy tzw. "agenta" zdolnego do wykonywania operacji na ramkach danych pandas. Agent potrafi interpretować pytania w języku naturalnym i przekształcać je na odpowiednie operacje i analizy na danych.

```python
from langchain.agents import AgentType
```
Stałe definiujące typy agentów dostępnych w LangChain. Różne typy agentów mają różne metody rozumowania i działania. Import sugeruje, że będziesz musiał określić konkretny typ agenta podczas konfiguracji.

```python
from langchain_openai import ChatOpenAI
```
Adapter umożliwiający wykorzystanie modeli językowych OpenAI (takich jak GPT) jako silnika "myślącego" dla agenta. ChatOpenAI to interfejs komunikacyjny pomiędzy twoim kodem a API OpenAI.

In [ ]:
class CFG:
    model = "gpt-4o-mini"
    temp = 0.3

In [ ]:
api_key = userdata.get("openaivision")
os.environ["OPENAI_API_KEY"] = api_key

# Funkcje

In [ ]:
def create_data(nrows=300):
    # Generate dates
    start_date = datetime(2024, 1, 1)
    dates = [start_date + timedelta(days=i) for i in range(nrows)]

    # Define data categories
    makes = [
        "Toyota",
        "Honda",
        "Ford",
        "Chevrolet",
        "Nissan",
        "BMW",
        "Mercedes",
        "Audi",
        "Hyundai",
        "Kia",
    ]
    models = ["Sedan", "SUV", "Truck", "Hatchback", "Coupe", "Van"]
    colors = ["Red", "Blue", "Black", "White", "Silver", "Gray", "Green"]

    # Create the dataset
    data = {
        "Date": dates,
        "Make": np.random.choice(makes, nrows),
        "Model": np.random.choice(models, nrows),
        "Color": np.random.choice(colors, nrows),
        "Year": np.random.randint(2015, 2023, nrows),
        "Price": np.random.uniform(20000, 80000, nrows).round(2),
        "Mileage": np.random.uniform(0, 100000, nrows).round(0),
        "EngineSize": np.random.choice([1.6, 2.0, 2.5, 3.0, 3.5, 4.0], nrows),
        "FuelEfficiency": np.random.uniform(20, 40, nrows).round(1),
        "SalesPerson": np.random.choice(
            ["Alice", "Bob", "Charlie", "David", "Eva"], nrows
        ),
    }

    # Create DataFrame and sort by date
    df = pd.DataFrame(data).sort_values("Date")

    return df


Ta funkcja `create_data` służy do generowania sztucznych danych o samochodach, które mogą być wykorzystane do testowania narzędzi analitycznych lub demonstracji działania agenta LangChain na przykładowych danych.

In [ ]:
def ask_agent(agent, question):
    response = agent.run(
        {
            "input": question,
            "agent_scratchpad": f"Human: {question}\nAI: To answer this question, I need to use Python to analyze the dataframe. \
        I'll use the python_repl_ast tool.\n\nAction: python_repl_ast\nAction Input: ",
        }
    )
    print(f"Question: {question}")
    print(f"Answer: {response}")


# Dane

In [7]:
df = create_data(100)

In [8]:
print(df.head())

        Date       Make      Model  Color  Year     Price  Mileage  \
0 2024-01-01    Hyundai        SUV    Red  2019  59486.09  38588.0   
1 2024-01-02  Chevrolet      Coupe  White  2021  35775.87  24529.0   
2 2024-01-03   Mercedes      Sedan   Gray  2022  21329.28  76680.0   
3 2024-01-04       Ford  Hatchback   Gray  2020  50413.27  21943.0   
4 2024-01-05     Toyota      Coupe    Red  2020  48264.05  64086.0   

   EngineSize  FuelEfficiency SalesPerson  
0         3.5            27.8         Bob  
1         3.0            34.2       David  
2         1.6            35.0     Charlie  
3         1.6            22.0         Eva  
4         1.6            23.0         Eva  


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            100 non-null    datetime64[ns]
 1   Make            100 non-null    object        
 2   Model           100 non-null    object        
 3   Color           100 non-null    object        
 4   Year            100 non-null    int64         
 5   Price           100 non-null    float64       
 6   Mileage         100 non-null    float64       
 7   EngineSize      100 non-null    float64       
 8   FuelEfficiency  100 non-null    float64       
 9   SalesPerson     100 non-null    object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(4)
memory usage: 7.9+ KB


In [10]:
print(df.describe())

                      Date         Year         Price       Mileage  \
count                  100   100.000000    100.000000    100.000000   
mean   2024-02-19 12:00:00  2018.520000  49939.428200  50120.230000   
min    2024-01-01 00:00:00  2015.000000  20391.160000    162.000000   
25%    2024-01-25 18:00:00  2016.000000  36018.997500  25863.500000   
50%    2024-02-19 12:00:00  2019.000000  47592.320000  51324.000000   
75%    2024-03-15 06:00:00  2021.000000  64021.165000  72416.250000   
max    2024-04-09 00:00:00  2022.000000  79610.280000  99014.000000   
std                    NaN     2.337572  17683.365436  28254.456823   

       EngineSize  FuelEfficiency  
count  100.000000      100.000000  
mean     2.593000       29.921000  
min      1.600000       20.300000  
25%      1.600000       24.675000  
50%      2.500000       30.050000  
75%      3.500000       35.200000  
max      4.000000       39.700000  
std      0.867861        5.738498  


# Agent

In [ ]:
muh_agent = create_pandas_dataframe_agent(
    ChatOpenAI(model=CFG.model, temperature=CFG.temp),
    df,
    verbose=True,
    allow_dangerous_code=True,
    agent_type=AgentType.OPENAI_FUNCTIONS,
)

Ta linia kodu tworzy agenta opartego na modelu językowym, który potrafi analizować dane w ramce danych pandas. Przyjrzyjmy się szczegółowo, jak ten agent jest konstruowany i jakie ma możliwości.

Funkcja `create_pandas_dataframe_agent()` to specjalna funkcja z biblioteki LangChain, która tworzy agenta wyspecjalizowanego w pracy z danymi tabularnymi. Taki agent potrafi interpretować pytania w języku naturalnym i przekształcać je na operacje analizy danych wykonywane na ramce danych.

Do tej funkcji przekazujemy kilka istotnych argumentów:

1. `ChatOpenAI(model = CFG.model, temperature = CFG.temp)` - Ten argument określa, jaki model językowy będzie "mózgiem" naszego agenta. W tym przypadku używamy modelu ChatOpenAI, który został skonfigurowany z:
   - Modelem `gpt-4o-mini` (zdefiniowanym wcześniej w klasie CFG)
   - Temperaturą 0.3 (również zdefiniowaną w CFG)
   
   Niska temperatura (0.3) sprawia, że odpowiedzi modelu będą bardziej deterministyczne i skoncentrowane na faktach, co jest pożądane przy analizie danych.

2. `df` - To nasza ramka danych, która została wcześniej utworzona funkcją `create_data()`. Zawiera ona dane o samochodach, które agent będzie analizował. Agent uzyskuje pełny dostęp do tej ramki danych i może wykonywać na niej operacje analizy.

3. `verbose=True` - Ten parametr włącza tryb "gadatliwy", co oznacza, że agent będzie pokazywał swój wewnętrzny tok rozumowania podczas analizy. Jest to bardzo pomocne do zrozumienia, jak agent interpretuje pytania i jakie kroki podejmuje, aby na nie odpowiedzieć.

4. `allow_dangerous_code = True` - Ten parametr pozwala agentowi na wykonywanie potencjalnie niebezpiecznego kodu. W standardowym ustawieniu agent ma ograniczenia dotyczące tego, jaki kod może wykonać, aby zapobiec przypadkowemu lub złośliwemu uszkodzeniu danych lub systemu. Włączenie tej opcji daje agentowi więcej swobody, ale może stanowić ryzyko bezpieczeństwa w środowiskach produkcyjnych.

5. `agent_type=AgentType.OPENAI_FUNCTIONS` - Ten parametr określa konkretny typ agenta, który będzie wykorzystywany. `OPENAI_FUNCTIONS` to typ agenta specjalnie zaprojektowany do współpracy z funkcjami modeli OpenAI, co pozwala na bardziej strukturyzowane interakcje i lepsze wykorzystanie możliwości modelu językowego.

Cały ten kod tworzy zaawansowanego agenta AI, który:
- Rozumie pytania zadawane w języku naturalnym
- Może analizować dane o samochodach w ramce danych
- Wykonuje zaawansowane operacje analizy danych przy użyciu bibliotek pandas i numpy
- Generuje odpowiedzi w języku naturalnym na podstawie wyników analizy
- Pokazuje szczegółowy proces myślowy podczas analizy (dzięki `verbose=True`)

Utworzony agent jest przypisany do zmiennej `muh_agent`, dzięki czemu można się do niego odwoływać w dalszych częściach kodu, na przykład przy użyciu wcześniej zdefiniowanej funkcji `ask_agent()`, żeby zadać mu pytania dotyczące danych.

In [ ]:
ask_agent(muh_agent, "What are the column names in this dataset?")

<ipython-input-6-7771c7f9e913>:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = agent.run({




> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'df.columns.tolist()'}`


['Date', 'Make', 'Model', 'Color', 'Year', 'Price', 'Mileage', 'EngineSize', 'FuelEfficiency', 'SalesPerson']The column names in the dataset are:

1. Date
2. Make
3. Model
4. Color
5. Year
6. Price
7. Mileage
8. EngineSize
9. FuelEfficiency
10. SalesPerson

> Finished chain.
Question: What are the column names in this dataset?
Answer: The column names in the dataset are:

1. Date
2. Make
3. Model
4. Color
5. Year
6. Price
7. Mileage
8. EngineSize
9. FuelEfficiency
10. SalesPerson


In [ ]:
ask_agent(muh_agent, "How many rows are in this dataset?")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'len(df)'}`


100The dataset contains 100 rows.

> Finished chain.
Question: How many rows are in this dataset?
Answer: The dataset contains 100 rows.


In [14]:
ask_agent(muh_agent, "What is the average price of cars sold?")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df['Price'].mean()"}`


49939.428199999995The average price of cars sold is approximately $49,939.43.

> Finished chain.
Question: What is the average price of cars sold?
Answer: The average price of cars sold is approximately $49,939.43.
